# First notebook

This tutorial is for users who can read basic Python and run notebook cells. It introduces the xarray concepts that the workflow uses.

Before you start, complete the {ref}`installation verification <verify-installation>`. Run this notebook in the same ERLabPy environment.

You will follow one analysis path from simulated angle-resolved data to momentum space and dispersion plots. The fixed simulation makes each result reproducible. You can compare your output with the output on this page.

## Analysis path

1. Create a three-dimensional angle-resolved dataset.
2. Inspect its xarray dimensions, coordinates, and metadata.
3. Crop the dataset by coordinate values.
4. Select a constant energy surface and average an energy interval.
5. Plot constant energy surfaces and energy–angle cuts.
6. Check the momentum conversion parameters and set the normal emission position.
7. Convert the angle-resolved data to momentum space and plot constant energy surfaces and energy–momentum cuts.
8. Overlay the Brillouin zone and extract K–M–K′ and Γ–M–K–Γ cuts.

## Imports

Import NumPy, xarray, Matplotlib, and ERLabPy:

- {mod}`numpy` supplies the reciprocal-space calculations used below.
- {mod}`xarray` supplies labeled arrays and coordinate-based operations.
- {mod}`matplotlib.pyplot` creates figures and axes.
- {mod}`erlab.analysis` supplies the path-interpolation function.
- {mod}`erlab.plotting` adds plotting helpers for common ARPES views.
- {func}`erlab.io.exampledata.generate_data_angles` generates reproducible simulated ARPES data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import erlab.analysis as era
import erlab.plotting as eplt
from erlab.io.exampledata import generate_data_angles

In [ ]:
%config InlineBackend.figure_formats = ["svg", "pdf"]
plt.rcParams["figure.dpi"] = 96

_ = xr.set_options(display_expand_data=False, keep_attrs=True)

## Angle-resolved example data

{func}`erlab.io.exampledata.generate_data_angles` creates a simulated three-dimensional {class}`xarray.DataArray`. It contains intensity as a function of `alpha`, `beta`, and `eV`; it is not a single energy–angle cut.

`assign_attributes=True` adds the metadata used for momentum conversion. The fixed seed makes the simulated noise reproducible.

The generated array has dimension order (`alpha`, `beta`, `eV`). {meth}`xarray.DataArray.transpose` changes it to (`eV`, `beta`, `alpha`). Transposition does not change the coordinate or intensity values.

In [ ]:
example_map = generate_data_angles(
    assign_attributes=True,
    seed=1,
).transpose("eV", "beta", "alpha")
example_map

## Labeled data and metadata

A {class}`xarray.DataArray` stores one array together with labels and metadata. These parts have different roles:

- **Dimensions** name the array axes. Their order describes the stored layout.
- **Dimension coordinates** give a physical value for each position on an axis.
- **Scalar coordinates** record a fixed condition that applies to the complete array.
- **Attributes** describe the array but do not take part in coordinate selection or alignment.

The DataArray display shows all four parts. {attr}`xarray.DataArray.sizes` gives the number of points on each named dimension.

In [ ]:
example_map.sizes

{attr}`xarray.DataArray.coords` contains the coordinate labels. Here, `eV`, `beta`, and `alpha` label dimensions. Coordinates such as `hv`, `xi`, and `delta` are scalars because they do not vary across this map.

In [ ]:
example_map.coords

{attr}`xarray.DataArray.attrs` contains descriptive metadata. ERLabPy accessors can read known attributes even though xarray does not use them for label-based selection.

In [ ]:
example_map.attrs

The energy range is -0.45 to 0.12 eV. Both angle ranges are -15° to 15°.

The names and units follow ERLabPy's {ref}`ARPES data conventions <data-conventions>`:

| Name | Role in this map | Physical meaning |
| --- | --- | --- |
| `eV` | Dimension coordinate | Energy relative to the Fermi level, in eV. ERLabPy uses negative values for occupied states. |
| `alpha` | Dimension coordinate | Emission angle measured along the analyzer slit. |
| `beta` | Dimension coordinate | Polar mapping angle for this Type 1 configuration. |
| `hv` | Scalar coordinate | Photon energy in eV. |
| `configuration` | Attribute | Experimental geometry used for momentum conversion. |
| `sample_workfunction` | Attribute | Work function in eV used for momentum conversion. |

The additional scalar coordinates describe fixed sample angles. `sample_temp` records the simulated sample temperature.

## Coordinate selection and averaging

{meth}`xarray.DataArray.sel` selects data by coordinate labels. {meth}`xarray.DataArray.isel` instead selects integer array positions. This tutorial uses coordinate labels because their physical meaning remains stable after cropping or transposition.

A {class}`slice` passed to `sel` crops a coordinate interval and keeps that dimension. It does not average the selected points. Create one working region that contains the energy range used below. Keep the full generated angular range:

In [ ]:
analysis_map = example_map.sel(eV=slice(-0.4, 0.1))
analysis_map

The result still has the `eV`, `beta`, and `alpha` dimensions. Only the `eV` size and range are smaller. The full `alpha` and `beta` ranges remain. The original coordinate labels and metadata remain attached.

A scalar selection removes one dimension. The requested coordinate does not have to occur exactly on a measured grid. Pass `method="nearest"` to select the closest sampled value:

In [ ]:
nearest_energy_map = analysis_map.sel(eV=-0.3, method="nearest")
nearest_energy_map

The result has `beta` and `alpha` dimensions. Its scalar `eV` coordinate contains the sampled energy that xarray selected.

One energy channel can contain more noise than an average over a finite energy range. The {meth}`xarray.DataArray.qsel` accessor selects by coordinate value and can average over that range. A `_width` argument gives the averaging width in the coordinate unit.

The first selection averages a 0.02 eV energy window centered at -0.3 eV. The second selects the `beta` value nearest to 5° without angular averaging. Each selected coordinate remains in the result as a scalar coordinate.

In [ ]:
angle_constant_energy = analysis_map.qsel(eV=-0.3, eV_width=0.02)
angle_energy_cut = analysis_map.qsel(beta=5.0)

angle_constant_energy

Before you display `angle_energy_cut`, predict its dimensions. Selecting one `beta` value removes that dimension. The result must keep `eV` and `alpha`, with the selected `beta` value stored as a scalar coordinate.

In [ ]:
angle_energy_cut

## Plotting two-dimensional data

Use Matplotlib to create the axes and ERLabPy to plot and annotate the data:

- {func}`matplotlib.pyplot.subplots` creates the figure and its axes.
- {func}`erlab.plotting.plot_array` draws one labeled two-dimensional DataArray.
- {func}`erlab.plotting.fermiline` marks the Fermi level at `eV=0`.
- {func}`erlab.plotting.set_titles` applies titles to several axes.

The dimension order determines the default plot axes. The surface has dimensions `beta` and `alpha`. The cut has dimensions `eV` and `alpha`. Use the same intensity color map for both panels. Mark the selected `beta` value on the constant energy surface and the averaged energy interval on the cut.

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(6.4, 2.8),
    layout="compressed",
    gridspec_kw={"width_ratios": (1.0, 1.35)},
)

eplt.plot_array(
    angle_constant_energy,
    ax=axes[0],
    cmap="Greys",
    gamma=0.7,
    aspect="equal",
)
axes[0].axhline(
    angle_energy_cut.beta,
    color="tab:red",
    linestyle="--",
    linewidth=0.8,
)

eplt.plot_array(
    angle_energy_cut,
    ax=axes[1],
    cmap="Greys",
    gamma=0.7,
)
axes[1].axhspan(-0.31, -0.29, color="tab:red", alpha=0.15)
axes[1].axhline(-0.31, color="tab:red", linestyle="--", linewidth=0.7)
axes[1].axhline(-0.29, color="tab:red", linestyle="--", linewidth=0.7)
eplt.fermiline(
    ax=axes[1],
    color="0.35",
    linestyle="--",
    linewidth=0.8,
)
eplt.set_titles(axes, ["Constant energy surface", "Energy–angle cut"])

The left panel has `alpha` on the horizontal axis and `beta` on the vertical axis. Both axes use degrees, so an equal aspect ratio is meaningful. The right panel has `alpha` on the horizontal axis and `eV` on the vertical axis. Its gray dashed line marks `eV=0`.

The red dashed line on the constant energy surface marks the selected `beta` value. The red band on the cut marks the 0.02 eV averaging interval used for the constant energy surface. `gamma` applies a power-law normalization to the displayed colors. It does not change the DataArray values.

## Constant energy surfaces

Use {func}`erlab.plotting.plot_slices` to compare constant energy surfaces at several energies. The function performs the requested {meth}`xarray.DataArray.qsel` selections and arranges the panels. Here, each panel uses the same 0.02 eV averaging width. Each panel has independent color limits so that changes in the contours remain visible.

In [ ]:
fig, axes = eplt.plot_slices(
    analysis_map,
    figsize=(6.4, 2.3),
    eV=[-0.4, -0.2, 0.0],
    eV_width=0.02,
    cmap="Greys",
    gamma=0.7,
    axis="image",
)

The automatic labels give the center energy of each panel. The constant energy contours contract as the selected energy approaches the Fermi level. The independent color limits show their shapes clearly. Do not use this figure to compare absolute intensities between panels.

:::{note}
If Qt is installed, use {meth}`xarray.DataArray.qshow` to inspect the working data in ImageTool from your local notebook:

```python
analysis_map.qshow()
```

This optional inspection does not change the data. It is not required for the remaining tutorial.
:::

## Momentum conversion parameters

The {class}`momentum accessor <erlab.accessors.kspace.MomentumAccessor>` reads the coordinates and metadata used by ERLabPy's momentum conversion functions.

Momentum conversion uses these parameters:

- The experimental configuration defines the relation between the measured angles and `kx` and `ky`.
- The photon energy and sample work function determine the photoelectron kinetic energy.
- The normal emission position sets zero in-plane momentum.

The generated data already contains the configuration, photon energy, and work function. You inspected these values in the coordinate and attribute displays above.

For this simulation, normal emission is `alpha=0.0` and `beta=0.0` by construction. ERLabPy does not infer this position from the simulated intensity. For measured data, determine normal emission from the measurement geometry and symmetry checks appropriate to the material.

Use {meth}`xarray.DataArray.copy` to preserve the angle-resolved working data. Then use {meth}`xarray.DataArray.kspace.set_normal` to store the known normal emission position. The method converts this position to the angular offsets for the selected configuration. {attr}`xarray.DataArray.kspace.offsets` shows the stored offsets.

In [ ]:
conversion_input = analysis_map.copy()
conversion_input.kspace.set_normal(alpha=0.0, beta=0.0)

conversion_input.kspace.offsets

{meth}`xarray.DataArray.kspace.convert` uses the momentum conversion functions and interpolates the intensity onto a regular momentum grid.

Here, ERLabPy selects the momentum bounds and grid spacing. {meth}`xarray.DataArray.transpose` orders the result as (`eV`, `ky`, `kx`).

In [ ]:
momentum_data = conversion_input.kspace.convert().transpose("eV", "ky", "kx")
momentum_data

The result should have dimensions `eV`, `ky`, and `kx`. The angular dimensions are no longer array axes. Both momentum ranges cross zero, and the momentum coordinates use Å⁻¹.

The output grid spacing controls interpolation sampling. It is not the experimental momentum resolution. Points on the rectangular grid that lie outside the measured angular coverage contain `NaN`.

## Conversion to momentum space

Select the same energy and averaging width before and after momentum conversion. Plot the two constant energy surfaces together. This comparison shows how the angular coordinates map to a regular `kx`, `ky` grid.

In [ ]:
momentum_constant_energy = momentum_data.qsel(eV=-0.3, eV_width=0.02)

fig, axes = plt.subplots(1, 2, figsize=(6.4, 3.0), layout="compressed")

eplt.plot_array(
    angle_constant_energy,
    ax=axes[0],
    cmap="Greys",
    gamma=0.7,
    aspect="equal",
)
eplt.plot_array(
    momentum_constant_energy,
    ax=axes[1],
    cmap="Greys",
    gamma=0.7,
    aspect="equal",
)
eplt.set_titles(axes, ["Angle coordinates", "Momentum coordinates"])

The left panel has `alpha` and `beta` axes in degrees. The right panel has `kx` and `ky` axes in Å⁻¹. Momentum conversion changes the coordinates and interpolates the intensity. It does not change the selected energy range. White regions outside the transformed angular coverage contain `NaN`.

## Slices of multidimensional data

Use {func}`erlab.plotting.plot_slices` to make one summary figure from the converted data. The first row contains constant energy surfaces. The second row contains energy–momentum cuts at fixed `ky`. Each cut averages over a 0.04 Å⁻¹ window.

In [ ]:
fig, axes = plt.subplots(
    2,
    3,
    layout="compressed",
    sharex=True,
    sharey="row",
)

eplt.plot_slices(
    [momentum_data],
    eV=[-0.4, -0.2, 0.0],
    eV_width=0.02,
    cmap="Greys",
    gamma=0.7,
    axes=axes[0],
    axis="image",
)
eplt.plot_slices(
    [momentum_data],
    ky=[0.0, 0.1, 0.3],
    ky_width=0.04,
    cmap="Greys",
    gamma=0.7,
    axes=axes[1],
)
eplt.clean_labels(axes)

The constant energy surfaces show the change in contour shape. The cuts show how the dispersion changes with `ky`. Each panel has independent color limits.

## High-symmetry cuts

{func}`erlab.io.exampledata.generate_data_angles` uses a hexagonal tight-binding model with the default lattice constant $a=6.97$ Å.

### K–M–K′

Define the vertices of one Brillouin-zone edge. {func}`erlab.analysis.interpolate.slice_along_path` interpolates the converted data between these vertices. The K–M and M–K′ segments have equal length. Subtract $2\pi/(3a)$ from the path coordinate to express momentum relative to M.

In [ ]:
lattice_constant = 6.97
kmk_vertices = {
    "kx": [
        0.0,
        np.pi / (np.sqrt(3) * lattice_constant),
        2 * np.pi / (np.sqrt(3) * lattice_constant),
    ],
    "ky": [
        4 * np.pi / (3 * lattice_constant),
        np.pi / lattice_constant,
        2 * np.pi / (3 * lattice_constant),
    ],
}

kmk_cut = era.interpolate.slice_along_path(
    momentum_data,
    vertices=kmk_vertices,
    step_size=0.005,
)
kmk_m_position = 2 * np.pi / (3 * lattice_constant)
kmk_cut = kmk_cut.assign_coords(path=kmk_cut.path - kmk_m_position)
kmk_cut

In [ ]:
kmk_vertex_positions = [-kmk_m_position, 0.0, kmk_m_position]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(6.4, 3.2),
    layout="compressed",
    gridspec_kw={"width_ratios": (1.0, 1.35)},
)

kmk_energy_map = momentum_data.qsel(eV=-0.2, eV_width=0.02)
eplt.plot_array(
    kmk_energy_map,
    ax=axes[0],
    cmap="Greys",
    gamma=0.7,
    aspect="equal",
)
eplt.plot_hex_bz(
    a=lattice_constant,
    ax=axes[0],
    fill=False,
    edgecolor="0.35",
    linewidth=0.8,
)
axes[0].plot(
    kmk_vertices["kx"],
    kmk_vertices["ky"],
    color="tab:red",
    marker="o",
    markersize=3,
    linewidth=1.2,
)
axes[0].set_title(r"$E = E_F - 0.2$ eV")

eplt.plot_array(
    kmk_cut,
    ax=axes[1],
    cmap="Greys",
    gamma=0.7,
)
eplt.fermiline(ax=axes[1], linestyle="--", linewidth=0.8)
axes[1].axvline(0.0, color="0.5", linestyle="--", linewidth=0.8)
eplt.mark_points(
    kmk_vertex_positions,
    ["K", "M", "K′"],
    y=0.12,
    ax=axes[1],
)
axes[1].set(
    xlabel=r"$k - \mathrm{M}$ (Å$^{-1}$)",
    xlim=(kmk_vertex_positions[0] - 0.02, kmk_vertex_positions[-1] + 0.02),
)

The cut coordinate is zero at M. The left panel confirms that the selected line follows the Brillouin-zone edge.

### Γ–M–K–Γ

Define a multi-segment Γ–M–K–Γ path. {func}`erlab.analysis.interpolate.slice_along_path` interpolates the converted data at evenly spaced points along each segment. The result keeps `eV` and replaces the `kx` and `ky` dimensions with one cumulative `path` dimension.

In [ ]:
high_symmetry_vertices = {
    "kx": [
        0.0,
        2 * np.pi / (np.sqrt(3) * lattice_constant),
        2 * np.pi / (np.sqrt(3) * lattice_constant),
        0.0,
    ],
    "ky": [0.0, 0.0, 2 * np.pi / (3 * lattice_constant), 0.0],
}

high_symmetry_cut = era.interpolate.slice_along_path(
    momentum_data,
    vertices=high_symmetry_vertices,
    step_size=0.005,
)
high_symmetry_cut

The rendered figure shows the selected path on a constant energy surface and uses the path vertices as tick labels. For the complete path-overlay procedure, see {ref}`how-to-python-extract-path`. For labels inside or outside the axes, see {ref}`how-to-plotting-annotate-arpes-figure`.

In [ ]:
path_vertices = np.column_stack(
    [high_symmetry_vertices["kx"], high_symmetry_vertices["ky"]]
)
segment_lengths = np.linalg.norm(np.diff(path_vertices, axis=0), axis=1)
path_vertex_positions = np.concatenate(([0.0], np.cumsum(segment_lengths)))
path_energy_map = momentum_data.qsel(eV=-0.2, eV_width=0.02)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(6.4, 3.2),
    layout="compressed",
    gridspec_kw={"width_ratios": (1.0, 1.35)},
)

eplt.plot_array(
    path_energy_map,
    ax=axes[0],
    cmap="Greys",
    gamma=0.7,
    aspect="equal",
)
eplt.plot_hex_bz(
    a=lattice_constant,
    ax=axes[0],
    fill=False,
    edgecolor="0.35",
    linewidth=0.8,
)
axes[0].plot(
    high_symmetry_vertices["kx"],
    high_symmetry_vertices["ky"],
    color="tab:red",
    marker="o",
    markersize=3,
    linewidth=1.2,
)
axes[0].set_title(r"$E = E_F - 0.2$ eV")

eplt.plot_array(
    high_symmetry_cut,
    ax=axes[1],
    cmap="Greys",
    gamma=0.7,
)
eplt.fermiline(ax=axes[1], linestyle="--", linewidth=0.8)
for position in path_vertex_positions[1:-1]:
    axes[1].axvline(
        position,
        color="0.5",
        linestyle="--",
        linewidth=0.8,
    )
axes[1].set_xticks(
    path_vertex_positions,
    labels=["Γ", "M", "K", "Γ"],
)
axes[1].set(
    xlabel="",
    xlim=(path_vertex_positions[0] - 0.03, path_vertex_positions[-1] + 0.03),
)

The hexagon and the red path use the lattice constant of the simulated band. The right panel shows the Γ–M–K–Γ cut. The x-axis labels mark the exact path vertices.

The interpolation follows an ideal line through momentum space. It does not average intensity over a finite width perpendicular to the path.

You started with simulated angle-resolved ARPES data. You inspected its xarray structure, selected an energy–angle cut, averaged a constant energy surface, converted the data to momentum space, overlaid the Brillouin zone, and extracted K–M–K′ and Γ–M–K–Γ cuts.

## Next steps

For the underlying ideas, read the {doc}`Explanation overview <../../explanation/index>`, {doc}`ARPES data conventions <../../explanation/data-conventions>`, and {doc}`momentum conversion <../../explanation/momentum-conversion>`.

For work on your own data, use the {doc}`data loading and saving <../../how-to/python/loading-and-saving>`, {doc}`data inspection and selection <../../how-to/python/inspection-and-selection>`, {doc}`plotting <../../how-to/plotting/index>`, and {doc}`momentum conversion <../../how-to/python/momentum-conversion>` guides. Complete function and parameter descriptions are in the {doc}`Python API reference <../../reference>`.